# SO101 Joint State Publisher

Publishes a `sensor_msgs/JointState` message for the SO-101 robot arm
based on the official `so101_new_calib.urdf`.

| Joint | Lower (rad) | Upper (rad) | Neutral |
|-------------|-------------|-------------|---------|
| shoulder_pan | -1.91986 | 1.91986 | 0 |
| shoulder_lift | -1.74533 | 1.74533 | 0 |
| elbow_flex | -1.69 | 1.69 | 0 |
| wrist_flex | -1.65806 | 1.65806 | 0 |
| wrist_roll | -2.74385 | 2.84121 | 0 |
| gripper | -0.17453 | 1.74533 | 0 |

In [1]:
from IPython.display import display, IFrame

from utils import get_base_url

url_prefix = get_base_url()

ros2ws = f"{url_prefix}/proxy/9090".replace("https", "wss")
web_url = f"{url_prefix}/proxy/3000/play/so-arm101?ros2ws={ros2ws}"

print(web_url)
# display(IFrame(src=web_url, width="90%", height="600px"))


https://jupyter.intel4coro.de/user/yxzhan-binder-template-jzzyoxqd/proxy/3000/play/so-arm101?ros2ws=wss://jupyter.intel4coro.de/user/yxzhan-binder-template-jzzyoxqd/proxy/9090


In [5]:
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import JointState

JOINTS = [
    ("shoulder_pan",   -1.91986,  1.91986),
    ("shoulder_lift",  -1.74533,  1.74533),
    ("elbow_flex",     -1.69,     1.69),
    ("wrist_flex",     -1.65806,  1.65806),
    ("wrist_roll",     -2.74385,  2.84121),
    ("gripper",        -0.17453,  1.74533),
]

JOINT_NAMES = [j[0] for j in JOINTS]
NEUTRAL_POSITIONS = [0.0] * len(JOINTS)  # new_calib: zero at midpoint

print("Joint names:", JOINT_NAMES)
print("Neutral positions:", NEUTRAL_POSITIONS)

Joint names: ['shoulder_pan', 'shoulder_lift', 'elbow_flex', 'wrist_flex', 'wrist_roll', 'gripper']
Neutral positions: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [6]:
class JointStatePublisher(Node):
    def __init__(self, positions: list[float]):
        super().__init__("so101_joint_state_publisher")
        self.pub = self.create_publisher(JointState, "/joint_states", 10)
        self.positions = positions
        self._publish()
        self.timer = self.create_timer(0.1, self._publish)

    def _publish(self):
        msg = JointState()
        msg.header.stamp = self.get_clock().now().to_msg()
        msg.name = JOINT_NAMES
        msg.position = self.positions
        msg.velocity = [0.0] * len(JOINT_NAMES)
        msg.effort = [0.0] * len(JOINT_NAMES)
        self.pub.publish(msg)
        self.get_logger().info(
            "Published: " + str([round(p, 4) for p in self.positions])
        )


if not rclpy.ok():
    rclpy.init(args=None)

node = JointStatePublisher(NEUTRAL_POSITIONS)

try:
    for _ in range(5):
        rclpy.spin_once(node, timeout_sec=0.2)
except KeyboardInterrupt:
    pass
finally:
    node.destroy_node()
    rclpy.shutdown()

[INFO] [1774049950.635328762] [so101_joint_state_publisher]: Published: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[INFO] [1774049950.738226129] [so101_joint_state_publisher]: Published: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[INFO] [1774049950.838172108] [so101_joint_state_publisher]: Published: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[INFO] [1774049950.940370709] [so101_joint_state_publisher]: Published: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[INFO] [1774049951.040274475] [so101_joint_state_publisher]: Published: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[INFO] [1774049951.141332064] [so101_joint_state_publisher]: Published: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


## Publish a custom pose

Edit the `custom_positions` dict and run the cell to publish specific joint angles.
All values are in **radians** and must stay within the limits shown in the table above.

In [7]:
custom_positions = {
    "shoulder_pan":   0.0,
    "shoulder_lift":  0.0,
    "elbow_flex":     0.0,
    "wrist_flex":     0.0,
    "wrist_roll":     0.0,
    "gripper":        0.0,
}

for name, lower, upper in JOINTS:
    val = custom_positions[name]
    assert lower <= val <= upper, f"{name}={val:.5f} outside [{lower}, {upper}]"

if not rclpy.ok():
    rclpy.init(args=None)

node = JointStatePublisher([custom_positions[n] for n in JOINT_NAMES])

try:
    for _ in range(5):
        rclpy.spin_once(node, timeout_sec=0.2)
except KeyboardInterrupt:
    pass
finally:
    node.destroy_node()
    rclpy.shutdown()

[INFO] [1774049954.395438076] [so101_joint_state_publisher]: Published: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[INFO] [1774049954.500402433] [so101_joint_state_publisher]: Published: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[INFO] [1774049954.600383676] [so101_joint_state_publisher]: Published: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[INFO] [1774049954.700433683] [so101_joint_state_publisher]: Published: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[INFO] [1774049954.800480665] [so101_joint_state_publisher]: Published: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[INFO] [1774049954.900443890] [so101_joint_state_publisher]: Published: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [8]:
import threading
import ipywidgets as widgets
from IPython.display import display

if not rclpy.ok():
    rclpy.init(args=None)

if "slider_node" not in globals():
    slider_node = rclpy.create_node("so101_slider_publisher")
    slider_pub = slider_node.create_publisher(JointState, "/joint_states", 10)

def publish_joint_state(positions: dict[str, float]):
    msg = JointState()
    msg.header.stamp = slider_node.get_clock().now().to_msg()
    msg.name = JOINT_NAMES
    msg.position = [positions[n] for n in JOINT_NAMES]
    msg.velocity = [0.0] * len(JOINT_NAMES)
    msg.effort = [0.0] * len(JOINT_NAMES)
    slider_pub.publish(msg)

sliders = {
    name: widgets.FloatSlider(
        value=0.0,
        min=lower,
        max=upper,
        step=0.001,
        description=name,
        continuous_update=True,
        style={"description_width": "120px"},
        layout=widgets.Layout(width="550px"),
    )
    for name, lower, upper in JOINTS
}

def on_change(change):
    publish_joint_state({n: s.value for n, s in sliders.items()})

for s in sliders.values():
    s.observe(on_change, names="value")

reset_btn = widgets.Button(description="Reset to neutral", button_style="info")

def on_reset(_):
    for (name, lower, upper), s in zip(JOINTS, sliders.values()):
        s.value = 0.0

reset_btn.on_click(on_reset)

display(widgets.VBox(list(sliders.values()) + [reset_btn]))